# Workflow and Hyperparameter Optimization

In [1]:
import pandas as pd
import seaborn as sns
import numpy as np

🏠 Import the house prices dataset. For simplicity, we will only keep numerical features.

🎯 Your goal will be to fit the best KNN Regressor. Specifically, how many "neighbors" (<font color=blue>K</font> in <font color=blue>K</font>NN) should you consider to get the best predictions for your house prices?

In [2]:
# Load raw data
data = pd.read_csv('https://d32aokrjazspmn.cloudfront.net/materials/houses_train_raw.csv', index_col="Id")

# Only keep numerical columns and raws without NaN
data = data.select_dtypes(include=np.number).dropna()

data

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SalePrice
Id,,,,,,,,,,,,,,,,,,,,,
1,60,65.0,8450,7,5,2003,2003,196.0,706,0,...,0,61,0,0,0,0,0,2,2008,208500
2,20,80.0,9600,6,8,1976,1976,0.0,978,0,...,298,0,0,0,0,0,0,5,2007,181500
3,60,68.0,11250,7,5,2001,2002,162.0,486,0,...,0,42,0,0,0,0,0,9,2008,223500
4,70,60.0,9550,7,5,1915,1970,0.0,216,0,...,0,35,272,0,0,0,0,2,2006,140000
5,60,84.0,14260,8,5,2000,2000,350.0,655,0,...,192,84,0,0,0,0,0,12,2008,250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1456,60,62.0,7917,6,5,1999,2000,0.0,0,0,...,0,40,0,0,0,0,0,8,2007,175000
1457,20,85.0,13175,6,6,1978,1988,119.0,790,163,...,349,0,0,0,0,0,0,2,2010,210000
1458,70,66.0,9042,7,9,1941,2006,0.0,275,0,...,0,60,0,0,0,0,2500,5,2010,266500


In [3]:
X = data.drop(columns=['SalePrice'])
y = data['SalePrice']

## 1. Train/Test Split

❓ **Question (Holdout)**❓

👇 Split the dataset to create `X_train`, `X_test`, `y_train` and `y_test`. Use:
- `test_size=0.3`
- `random_state=0` to compare your results with a friend

In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

## 2. Scaling

⚖️ Scaling is always critically important for the KNN algorithm.

❓ **Question (Scaling)** ❓ 

* Scale your training set and test set.
* Here, let's simply apply `StandardScaler` and not waste time choosing a scaler per feature. Indeed, the objectives of this exercise are:
    * To review KNN
    * To understand GridSearchCV
    * To understand RandomizedSearchCV

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train)
X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns)

## 3. Baseline KNN Model

❓ **Question (A baseline for KNN)** ❓

Cross-validate (*cv = 5*) a simple KNN regressor that considers only _the nearest neighbor_ and compute the mean score across the 5 folds.

In [6]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_validate

knn = KNeighborsRegressor(n_neighbors=1)
cv_scores = cross_validate(knn, X_train_scaled, y_train, cv=5)["test_score"].mean()


## 4. GridSearch

### 4.1. First GridSearch

❓ **Question (GridSearch v1)**❓

Let's use SKLearn `GridSearchCV` to find the best KNN hyperparameter `n_neighbors`.
- Start with a coarse-grained approach using `n_neighbors` = [1,5,10,20,50]
- Cross-validate each parameter with 5 folds
- Make sure to maximize your performance time using `n_jobs`

In [7]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsRegressor

# Instantiate model
model = KNeighborsRegressor()

# Hyperparameter Grid
k_grid = {"n_neighbors": [1, 5, 10, 20, 50]}

# Instantiate Grid Search
grid_search = GridSearchCV(model, k_grid, cv=5, n_jobs=-1)

# Fit data to Grid Search
grid_search.fit(X_train_scaled, y_train)


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",KNeighborsRegressor()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'n_neighbors': [1, 5, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : the fold and candidate parameter index

❓ **Question (best parameters)** ❓

According to the GridSearch, what is the optimal value of K?

In [8]:
grid_search.best_params_

{'n_neighbors': 10}

❓ **Question (scoring)** ❓ What is the best score produced by the optimal value of K?

In [9]:
grid_search.best_score_

0.7596697382171873

### 4.2. Second GridSearch

❓ **Question (GridSearch V2)** ❓

Now, we have an idea of where the best $K$ is, but some values we haven't tried might result in better performance.

* Re-run the GridSearch by trying some values for $K$ around your previous best value
* What are the `best_score` and `best_k` for this refined GridSearch?

In [11]:
# Instantiate model
model = KNeighborsRegressor()

# Hyperparameter Grid
k_grid = {"n_neighbors": np.arange(11, 30, 1)}

# Instantiate Grid Search
grid_search = GridSearchCV(model, k_grid, cv=5, n_jobs=-1)

# Fit data to Grid Search
grid_search.fit(X_train_scaled, y_train)

# Get best parameters and best score
print(grid_search.best_params_["n_neighbors"])
print(grid_search.best_score_)

16
0.7666311417513013


In [13]:
best_k = grid_search.best_params_["n_neighbors"]
best_score = grid_search.best_score_

***🧪 Test your code***

In [14]:
from nbresult import ChallengeResult
result = ChallengeResult('knn',
                         best_k=best_k,
                         best_score=best_score)
result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/yaren/.pyenv/versions/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/yaren/code/ds_projects/data-workflow/tests
plugins: anyio-4.12.1, dash-4.0.0, typeguard-4.4.2
collecting ... collected 2 items

test_knn.py::TestKnn::test_best_k PASSED                                 [ 50%]
test_knn.py::TestKnn::test_best_score PASSED                             [100%]

============================== 2 passed in 0.07s ===============================


💯 You can commit your code:

git add tests/knn.pickle

git commit -m 'Completed knn step'

git push origin master



### 4.3. Visual Check (Manual GridSearch)

☝️ This problem is actually simple enough to GridSearch manually.

❓ **Question (Manual GridSearch)** ❓

- Manually loop over all values of $K$ from $1$ to $50$ and store the mean cross-validated scores of each model in a list.
- Plot the scores as a function of $K$ to visually find the best $K$ using the `Elbow Method`

In [ ]:
# YOUR CODE HERE

❓ Can you guess what makes GridSearchCV a better option than such a manual loop?

<details>
    <summary>Answer</summary>

- Sklearn's `n_jobs=-1` option allows you to parallelize the search, using all your CPU cores
- What if you had multiple hyperparameters to optimize together?
</details>

## 5. GridSearch with Multiple Parameters

👩🏻‍🏫 KNNRegressor supports various _distance metrics_ through the `p` hyperparameter

📚 [sklearn.neighbors.KNeighborsRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html)

❓ **Question (tuning multiple parameters)** ❓

* Use GridSearchCV to search for the best $K$ and $p$ simultaneously.
    * Try all combinations for $K = [1, 5, 10, 20, 50]$ and $p = [1, 2, 3]$.

In [ ]:
# YOUR CODE HERE

❓ **Question (number of sub-models)**❓

How many sub-models did you train in total?

<details>
    <summary>Hint</summary>

Much more than 15. Think twice :)
    <details>
    <summary>Answer</summary>

75 models because CV=5
</details>

In [ ]:
# YOUR CODE HERE

❓ **Question (best parameters and best score after tuning the model with multiple parameters)**❓

What are the *best parameters* and the *best score*?

In [ ]:
# YOUR CODE HERE

## 6. Random Search

Now let's see if a RandomizedSearch can find a better combination by fitting the same number of models.

❓ **Question (RandomizedSearchCV)** ❓

Using `RandomizedSearchCV`:
- Randomly sample $K$ from a uniform `scipy.stats.randint(1,50)` ([docs](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.randint.html)) distribution
- Sample $p$ from the list $[1,2,3]$
- Use the correct `n_iter` and `cv` numbers to fit exactly the same number of models as in your previous GridSearchCV.

In [ ]:
# YOUR CODE HERE

## 7. Generalization

❓ **Question (fine-tuning your model one more time)**❓

- Refine your RandomizedSearchCV if you wish
- Select your best model

In [ ]:
# YOUR CODE HERE

Now try displaying your `cv_results` as a `DataFrame` — it will help you visualize what's happening inside the CV! 😉

In [ ]:
# YOUR CODE HERE

❓ **Question (Evaluating the "best" model)** ❓

* It's time to explore the performance of our model with the "best parameters" on the **unseen** test set `X_test`.
    * Compute the r2 score on the test set and save it as `r2_test`.

In [ ]:
# YOUR CODE HERE

❓ **Question (Taking a step back)** ❓

Do you think the optimized model generalizes well?

<details><summary>Answer</summary>

The test score may decrease slightly compared to the training set. Probably not more than 5%. This could be due to:
- A non-representative train/test split
- Too small a number of cross-validation folds during the model tuning phase, which leads to overfitting. The more cross-validation you do, the more robustly your findings generalize — but if your dataset is very small, you can't increase cv too much because you won't have enough observations to be representative in each fold.
- Our dataset is very small and our hyperparameter optimization is therefore highly dependent on (and overfitting to) our train/test split. Always make sure your dataset is much larger than the total number of hyperparameter combinations you tried!
    
</details>

***🧪 Test your code***

In [ ]:
from nbresult import ChallengeResult
result = ChallengeResult('r2', 
                         r2_test=r2_test)
result.write()
print(result.check())